In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.ensemble import RandomForestRegressor


def _feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    side_num = (df["side"].values == "upper").astype(float) if "side" in df.columns else (df["y"].values >= 0).astype(float)
    return pd.DataFrame({"x": df["x"].values, "y": df["y"].values, "Alpha": df["Alpha"].values, "side_upper": side_num})


def train_interpretable_proxy(train_df: pd.DataFrame, model_predictions: np.ndarray | None = None, random_state: int = 42):
    """Train a tree proxy for model-agnostic SHAP/permutation analysis.

    If model_predictions are supplied, the proxy explains the surrogate behavior.
    Otherwise it explains the measured Cp values in the training data.
    """
    X = _feature_frame(train_df)
    y = model_predictions if model_predictions is not None else train_df["Cp"].values
    proxy = RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=random_state, n_jobs=-1)
    proxy.fit(X, y)
    return proxy, X


def permutation_importance_table(model, eval_df: pd.DataFrame, target_col: str = "Cp", n_repeats: int = 30, random_state: int = 42) -> pd.DataFrame:
    """Permutation feature importance using a fitted sklearn-style model."""
    X = _feature_frame(eval_df)
    y = eval_df[target_col].values
    scorer = make_scorer(mean_squared_error, greater_is_better=False)
    result = permutation_importance(model, X, y, scoring=scorer, n_repeats=n_repeats, random_state=random_state, n_jobs=-1)
    table = pd.DataFrame({
        "feature": X.columns,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }).sort_values("importance_mean", ascending=False)
    return table


def shap_importance_table(model, eval_df: pd.DataFrame) -> pd.DataFrame:
    """Mean absolute SHAP importance table for a fitted tree proxy.

    Requires the optional 'shap' package.
    """
    try:
        import shap
    except Exception as exc:  # pragma: no cover
        raise ImportError("Install shap to use shap_importance_table: pip install shap") from exc
    X = _feature_frame(eval_df)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    values = np.abs(np.asarray(shap_values)).mean(axis=0)
    return pd.DataFrame({"feature": X.columns, "mean_abs_shap": values}).sort_values("mean_abs_shap", ascending=False)
